# 8.2 What can a CNN learn from the arrangement of pixels?

**Question:** Can we distinguish compact, linear and scattered patterns when they have identical average brightness?

**Objectives:** calculate one convolution response; explain weight sharing, feature maps, ReLU and pooling; train a small CNN; compare a mean-pixel baseline; test sensitivity to pixel shuffling.

**Prerequisites:** arrays and lesson 8.1's forward/loss/update concepts. **Time:** 90 minutes. Existing course dependencies suffice: this small CNN is implemented in NumPy, including learned convolution filters. No GPU, downloads or PyTorch installation is needed.

**Data:** invented 16×16 grayscale patterns, not satellite observations or real roads, forests or rivers. Group IDs represent independently generated synthetic scenes; variants of a scene remain together. Scores illustrate a deliberately easy spatial-pattern task.

## Why add convolution after an ANN?

A convolutional layer slides a small learned kernel across an image. It uses the **same weights** at every position: weight sharing. Each kernel produces a **feature map** of local responses. **ReLU** replaces negative responses with zero; **global average pooling** reduces each feature map to one average. A final linear layer produces three class scores, converted to probabilities by softmax.

Our model: `16×16×1 -> six 3×3 filters -> 14×14×6 -> ReLU -> six averages -> three classes`. Stride is 1, with no padding. There are only **81 trainable parameters**. Pooling reduces sensitivity to position but does not guarantee perfect translation invariance, especially at image boundaries.

What deep-learning libraries call convolution generally uses cross-correlation (the kernel is not flipped). We follow that convention, documented in [PyTorch Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). This is image classification, not pixelwise segmentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, ConfusionMatrixDisplay
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course.py').exists())
OUT = ROOT/'data'/'generated'/'cnn'
OUT.mkdir(parents=True, exist_ok=True)
NAMES = ['compact', 'linear', 'scattered']
rng = np.random.default_rng(42)
images, labels, groups = [], [], []
for scene in range(90):
    # Scene brightness is shared by all classes, eliminating class/brightness confounding.
    background, contrast = rng.uniform(0.02, 0.12), rng.uniform(0.65, 0.85)
    for label in range(3):
        patch = np.zeros((16, 16))
        if label == 0:
            row, col = rng.integers(2, 11, size=2)
            patch[row:row+4, col:col+4] = 1
        elif label == 1:
            line = int(rng.integers(2, 14))
            if rng.random() < 0.5: patch[line, :] = 1
            else: patch[:, line] = 1
        else:
            patch.flat[rng.choice(256, 16, replace=False)] = 1
        for view in [patch, np.rot90(patch)]:
            images.append(background + contrast * view)
            labels.append(label); groups.append(scene)
X = np.asarray(images, dtype=float)
y, scene_ids = np.asarray(labels), np.asarray(groups)
order = rng.permutation(90)
train = np.isin(scene_ids, order[:60])
validation = np.isin(scene_ids, order[60:75])
test = np.isin(scene_ids, order[75:])
assert not (set(scene_ids[train]) & set(scene_ids[test]))
assert not (set(scene_ids[train]) & set(scene_ids[validation]))
assert not (set(scene_ids[validation]) & set(scene_ids[test]))
assert np.all(train.astype(int) + validation.astype(int) + test.astype(int) == 1)
# Every class has exactly 16 bright pixels per image and matching means within a scene.
for scene in range(90):
    assert np.ptp(X[scene_ids == scene].mean(axis=(1, 2))) < 1e-12
print('Images by split:', [int(mask.sum()) for mask in [train, validation, test]])
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for label, ax in enumerate(axes):
    ax.imshow(X[np.flatnonzero(train & (y == label))[0]], cmap='gray', vmin=0, vmax=1)
    ax.set_title(NAMES[label]); ax.axis('off')
fig.suptitle('Synthetic patterns: equal bright-pixel counts'); fig.tight_layout(); plt.show()

## Work through one filter by hand

Multiply a 3×3 neighborhood by the kernel elementwise, sum the nine products, and add a bias. Move one pixel and repeat. The hand-set kernel below detects one kind of contrast; it is not one of the learned filters yet. Predict the sign of its response before running.

In [ ]:
example = X[np.flatnonzero(train & (y == 1))[0]]
kernel = np.array([[-1., -1., -1.], [2., 2., 2.], [-1., -1., -1.]])
windows = sliding_window_view(example, (3, 3))
response = np.einsum('ijab,ab->ij', windows, kernel)
row, col = np.unravel_index(np.argmax(np.abs(response)), response.shape)
products = windows[row, col] * kernel
display(pd.DataFrame(products))
print('Sum of nine products:', products.sum(), '| feature-map value:', response[row, col])
assert np.isclose(products.sum(), response[row, col])
print('Shapes: image', example.shape, '→ response', response.shape)
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, data, title in zip(axes, [example, kernel, response], ['Input', 'Hand-set kernel', 'Response']):
    ax.imshow(data, cmap='coolwarm'); ax.set_title(title); ax.axis('off')
fig.tight_layout(); plt.show()

## Implement the tiny CNN

We flatten each 3×3 window into nine numbers and apply six shared filters. `einsum` performs the weighted sums at every position. Global averaging yields six features per image; the classifier then produces three logits. Softmax is computed after subtracting the maximum score for stability.

The backward function propagates cross-entropy error through the classifier, average pooling and ReLU to the filters. Thus **both filters and classifier weights learn**. There is no earlier trainable layer, so input-image gradients are unnecessary. Beginners can focus on the shapes and the forward pass; the gradient code is available for inspection.

In [ ]:
def make_windows(images):
    return sliding_window_view(images, (3, 3), axis=(1, 2)).reshape(len(images), 196, 9)

def initialize(seed=7):
    random = np.random.default_rng(seed)
    return {'K': random.normal(0, 0.3, (9, 6)), 'b': np.full(6, 0.05),
            'W': random.normal(0, 0.3, (6, 3)), 'c': np.zeros(3)}

def forward(windows, p):
    z = np.einsum('nlj,jf->nlf', windows, p['K']) + p['b']
    pooled = np.maximum(z, 0).mean(axis=1)
    logits = pooled @ p['W'] + p['c']
    shifted = logits - logits.max(axis=1, keepdims=True)
    log_probability = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    return z, pooled, log_probability

def loss(windows, labels, p):
    return float(-forward(windows, p)[2][np.arange(len(labels)), labels].mean())

def gradients(windows, labels, p):
    z, pooled, log_probability = forward(windows, p)
    error = np.exp(log_probability)
    error[np.arange(len(labels)), labels] -= 1
    error /= len(labels)
    dz = ((error @ p['W'].T)[:, None, :] / windows.shape[1]) * (z > 0)
    return {'K': np.einsum('nlj,nlf->jf', windows, dz), 'b': dz.sum(axis=(0, 1)),
            'W': pooled.T @ error, 'c': error.sum(axis=0)}

patches = make_windows(X)
initial = initialize()
assert sum(a.size for a in initial.values()) == 81
z, pooled, logp = forward(patches[:2], initial)
assert z.shape == (2, 196, 6) and pooled.shape == (2, 6) and logp.shape == (2, 3)
assert np.allclose(np.exp(logp).sum(axis=1), 1)
# Numerical gradient check: validate one element of each learned parameter array.
batch = np.flatnonzero(train)[:6]
analytic = gradients(patches[batch], y[batch], initial)
for name in initial:
    plus = {k: v.copy() for k, v in initial.items()}; minus = {k: v.copy() for k, v in initial.items()}
    plus[name].flat[0] += 1e-5; minus[name].flat[0] -= 1e-5
    numerical = (loss(patches[batch], y[batch], plus) - loss(patches[batch], y[batch], minus)) / 2e-5
    assert np.isclose(numerical, analytic[name].flat[0], atol=1e-6, rtol=1e-3), name
print('Forward-shape, probability and gradient checks passed.')

## Train and select using held-out scenes

Use mini-batches of 60 images for 120 epochs. Adam uses running averages of gradients and squared gradients to scale updates; unlike lesson 8.1's basic gradient descent, different weights receive adaptive step sizes. The implementation is short and visible below.

Only training images contribute gradients. After each epoch, save a copy of the parameters with the lowest validation loss. Scene-group separation prevents rotated variants from crossing partitions. The fixed test scenes remain untouched until evaluation. No per-dataset normalization is needed for these intentionally bounded 0–1 inputs.

In [ ]:
def train_cnn(epochs=120, learning_rate=0.02, seed=7):
    p = initialize(seed)
    first = {k: np.zeros_like(v) for k, v in p.items()}
    second = {k: np.zeros_like(v) for k, v in p.items()}
    random = np.random.default_rng(seed)
    history, best, best_loss, step = [], None, np.inf, 0
    training_ids = np.flatnonzero(train)
    for epoch in range(epochs + 1):
        train_loss = loss(patches[train], y[train], p)
        val_loss = loss(patches[validation], y[validation], p)
        assert np.isfinite([train_loss, val_loss]).all()
        history.append({'epoch': epoch, 'train_loss': train_loss, 'validation_loss': val_loss})
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch
            best = {k: v.copy() for k, v in p.items()}
        if epoch == epochs: break
        order = random.permutation(training_ids)
        for start in range(0, len(order), 60):
            ids = order[start:start+60]
            derivative = gradients(patches[ids], y[ids], p)
            step += 1
            for name in p:
                first[name] = 0.9 * first[name] + 0.1 * derivative[name]
                second[name] = 0.999 * second[name] + 0.001 * derivative[name]**2
                corrected_first = first[name] / (1 - 0.9**step)
                corrected_second = second[name] / (1 - 0.999**step)
                p[name] -= learning_rate * corrected_first / (np.sqrt(corrected_second) + 1e-8)
    return best, pd.DataFrame(history), best_epoch

model, history, best_epoch = train_cnn()
assert not np.allclose(model['K'], initial['K']), 'Convolution filters must actually learn.'
assert history.train_loss.iloc[-1] < history.train_loss.iloc[0]
print('Selected validation checkpoint:', best_epoch)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.epoch, history.train_loss, label='training')
ax.plot(history.epoch, history.validation_loss, label='validation')
ax.axvline(best_epoch, color='gray', linestyle='--', label='selected epoch')
ax.set(xlabel='Epoch', ylabel='Cross-entropy', title='Learning spatial patterns')
ax.legend(); fig.tight_layout(); fig.savefig(OUT / 'learning_curves.png', dpi=140); plt.show()
history.to_csv(OUT/'loss_history.csv', index=False)

## Evaluate and challenge the spatial information

The mean-pixel logistic-regression baseline receives one number per image. Every class shares the same brightness distribution by construction, so this statistic cannot distinguish our labels.

Now randomly permute pixels within each test image. This preserves its pixel values, histogram and mean but destroys most spatial arrangement. Keep the original labels only to measure how much the trained classifier's behavior changes. The shuffled images no longer necessarily belong to the original shape classes: this is a **perturbation diagnostic**, not an ordinary accuracy benchmark or proof of causality. Do not tune the model after inspecting test results.

In [ ]:
means = X.mean(axis=(1, 2)).reshape(-1, 1)
baseline = LogisticRegression(random_state=42).fit(means[train], y[train])
test_images = X[test].copy()
shuffle_rng = np.random.default_rng(99)
shuffled = np.stack([shuffle_rng.permutation(image.ravel()).reshape(16, 16) for image in test_images])
assert np.allclose(shuffled.mean(axis=(1, 2)), means[test, 0])
assert np.allclose(np.sort(shuffled.reshape(-1, 256)), np.sort(test_images.reshape(-1, 256)))
original_prediction = forward(patches[test], model)[2].argmax(axis=1)
shuffled_prediction = forward(make_windows(shuffled), model)[2].argmax(axis=1)
assert np.array_equal(baseline.predict(means[test]), baseline.predict(shuffled.mean(axis=(1, 2)).reshape(-1, 1)))
scores = pd.DataFrame([
    {'condition': 'mean-pixel baseline', 'balanced_accuracy_vs_original_labels': balanced_accuracy_score(y[test], baseline.predict(means[test]))},
    {'condition': 'CNN: original held-out scenes', 'balanced_accuracy_vs_original_labels': balanced_accuracy_score(y[test], original_prediction)},
    {'condition': 'CNN: shuffled diagnostic', 'balanced_accuracy_vs_original_labels': balanced_accuracy_score(y[test], shuffled_prediction)}
])
display(scores)
print('Fraction of CNN predictions changed by shuffling:', np.mean(original_prediction != shuffled_prediction))
fig, ax = plt.subplots(figsize=(6, 4))
ConfusionMatrixDisplay.from_predictions(y[test], original_prediction, labels=[0, 1, 2],
                                       display_labels=NAMES, colorbar=False, ax=ax)
ax.set_title('Original held-out scenes'); fig.tight_layout(); plt.show()
scores.to_csv(OUT/'evaluation.csv', index=False)

## Look inside the learned layer

Select a held-out linear pattern. Compare its learned filters and post-ReLU feature maps. A bright response indicates activation, not proof that a unit represents a named object. Use one shared scale across filters and another shared scale across maps so differences remain comparable.

In [ ]:
index = np.flatnonzero(test & (y == 1))[0]
activations = np.maximum(forward(patches[index:index+1], model)[0], 0)[0].reshape(14, 14, 6)
filters = model['K'].reshape(3, 3, 6)
limit = np.abs(filters).max()
fig, axes = plt.subplots(2, 7, figsize=(15, 5), constrained_layout=True)
axes[0, 0].imshow(X[index], cmap='gray', vmin=0, vmax=1); axes[0, 0].set_title('Input: linear')
relative = int(np.flatnonzero(np.flatnonzero(test) == index)[0])
axes[1, 0].imshow(shuffled[relative], cmap='gray', vmin=0, vmax=1); axes[1, 0].set_title('Shuffled input')
for f in range(6):
    kernel_plot = axes[0, f+1].imshow(filters[:, :, f], cmap='coolwarm', vmin=-limit, vmax=limit)
    axes[0, f+1].set_title(f'Kernel {f+1}')
    map_plot = axes[1, f+1].imshow(activations[:, :, f], cmap='magma', vmin=0, vmax=max(1e-8, activations.max()))
    axes[1, f+1].set_title(f'Feature map {f+1}')
for ax in axes.flat: ax.axis('off')
fig.colorbar(kernel_plot, ax=axes[0, 1:].tolist(), shrink=0.7, label='Weight')
fig.colorbar(map_plot, ax=axes[1, 1:].tolist(), shrink=0.7, label='ReLU response')
fig.suptitle('Learned CNN filters and responses • synthetic patterns')
fig.savefig(OUT/'feature_maps.png', dpi=140); plt.show()

## Your independent challenge

1. Predict how changing the learning rate to 0.005 affects loss at the same epoch budget. Run that one-factor experiment and compare validation curves only.
2. Shuffle **validation** images and report the prediction-change fraction. Verify that mean brightness stays unchanged. Explain why the mean-pixel baseline cannot detect this manipulation.
3. Choose a 3×3 neighborhood and calculate its response to one learned kernel, including the learned bias. Apply ReLU and verify it against the feature map.
4. Propose a real satellite task and explain the labels, channels, pixel size and scene/date grouping it would require. Why is this fixture insufficient evidence that a model can detect roads or rivers?

**Submit:** one hand-calculated response, a controlled validation comparison, a feature-map interpretation and a split design. Keep test results out of model selection.

In [ ]:
# Your experiment here. Use validation images for new diagnostics.

<details><summary>Solution guidance - open after trying</summary>

Call `train_cnn(learning_rate=0.005)` and compare the validation loss curve with the original run. A slower decrease at a fixed budget does not prove lower ultimate capacity. For a learned filter `f` and window position `(row, col)`, calculate `sum(window * model['K'][:, f].reshape(3,3)) + model['b'][f]`, then take `max(0, result)`. Mean-pixel predictions stay unchanged under a permutation because sums are invariant. Real evaluation must group patches from the same source acquisition or nearby overlapping areas and consider temporal/spatial buffers; scene IDs alone do not eliminate all dependence.

</details>

## Check yourself and assessment

Why do six kernels produce six feature maps? Why is each map 14×14? Which parameters are shared across positions? What spatial information does global pooling discard?

| Criterion | Points | Evidence |
|---|---:|---|
| Convolution understanding | 3 | Correct manual response, bias and ReLU; shape explanation |
| Learning and evaluation | 3 | Learned filters, scene-group split and mean-baseline interpretation |
| Independent experiment | 2 | One-factor validation comparison and shuffle diagnostic |
| Communication | 2 | Feature-map interpretation, synthetic disclosure and real-data limitations |

**Mastery target:** 8/10 with no overlapping scene variants across train/validation/test.

## Transfer to TorchGeo and satellite imagery

This is patch classification, not a land-cover map. For real imagery, define labels independently, preserve CRS/transform and channel identity, mask invalid pixels, choose consistent pixel support and split by acquisition/region/date. Normalization statistics must come from training data. Real channels need not be grayscale or bounded 0–1. Compare a simple baseline before a larger CNN and inspect regional/class-specific failures.

Return to [lesson 2.1](../02-pytorch-torchgeo/01_torchgeo_intro.ipynb) to connect raster windows to model tensors. The NumPy model here teaches the mechanics; a framework such as PyTorch provides automatic differentiation and optimized convolution for larger experiments. Learning from scratch here is different from the optional pretrained inference lab.

## Educator note

Suggested 90 minutes: 10 to predict from equal-brightness examples, 15 for hand convolution, 15 for network shapes, 15 for training/evaluation, 15 for feature maps and shuffling, and 20 for the independent challenge. Backpropagation and Adam code are available for advanced learners; beginners can focus on forward shapes and evidence that the filters changed.